In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

!pip install torch lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 8.5 MB/s eta 0:00:00


In [4]:
import os
import json
import copy
import warnings
import itertools

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from lifelines.utils import concordance_index

In [5]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
EXP_1_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/exp_1/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/"

os.makedirs(EXP_1_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = EXP_1_OUTPUT_DIR + "hcc_exp1_train_selected.csv"

TUNING_RESULTS_PATH = RESULTS_DIR + "hcc_exp1_deepsurv_tuning_results.csv"
TUNING_FOLD_DETAILS_PATH = RESULTS_DIR + "hcc_exp1_deepsurv_tuning_fold_details.csv"
BEST_PARAMS_JSON_PATH = RESULTS_DIR + "hcc_exp1_deepsurv_best_hyperparameters.json"
BEST_PARAMS_CSV_PATH = RESULTS_DIR + "hcc_exp1_deepsurv_best_hyperparameters.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

In [6]:
train_data = pd.read_csv(TRAIN_PATH)

print("Train data shape:", train_data.shape)
train_data.head()

Train data shape: (1085, 32)


,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2Ap16INK4A,SAMPLE_TYPE_Metastasis,...,C_N_,C_N_Yes,METASTASIS_Yes,MAJOR_LIVER_RESECTION_No,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_No,OS_MONTHS,OS_EVENT
0,886.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,82.296,0
1,755.0,6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,51.324,1
2,635.0,5.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,6.600,1
3,537.0,6.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,24.480,0
4,1056.0,3.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,14.592,1


In [7]:
print("Train missing values:", int(train_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

Train missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64


In [8]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"

print("ID leakage guard passed.")

ID leakage guard passed.


In [9]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)

train_data.shape

(1085, 32)

In [10]:
DEPENDENT_COLUMNS = ["OS_MONTHS", "OS_EVENT"]
ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]

GENES = [
    col for col in train_data.columns
    if col not in DEPENDENT_COLUMNS + ID_COLUMNS
]

print("Feature count:", len(GENES))
GENES[:10]

Feature count: 30


['SAMPLE_COVERAGE',
 'TUMOR_SIZE',
 'MUT_ALB',
 'MUT_ARID1A',
 'MUT_ARID1B',
 'MUT_UPF1',
 'MUT_WHSC1',
 'MUT_ZFHX3',
 'CNA_CDKN2Ap16INK4A',
 'SAMPLE_TYPE_Metastasis']

In [11]:
X = train_data[GENES].copy()
y = train_data[DEPENDENT_COLUMNS].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1085, 30)
y shape: (1085, 2)


In [12]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
RANDOM_STATE = 42

print("Device:", DEVICE)
print("N_SPLITS:", N_SPLITS)

Device: cpu
N_SPLITS: 5


In [13]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

In [14]:
class DeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name):
        super(DeepSurv, self).__init__()

        activation_map = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "SELU": nn.SELU
        }

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(activation_map[activation_name]())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            current_dim = hidden_dim

        self.feature_extractor = nn.Sequential(*layers)
        self.output_layer = nn.Linear(current_dim, 1)

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.output_layer(x)

In [15]:
def nll_loss(preds, durations, events):
    order = torch.argsort(durations, descending=True)

    hazards = preds[order].reshape(-1)
    events = events[order].reshape(-1)

    log_risk = torch.logcumsumexp(hazards, dim=0)
    uncensored_loss = hazards - log_risk
    censored_loss = uncensored_loss * events

    return -torch.sum(censored_loss) / torch.sum(events)

In [16]:
def build_optimizer(model, optimizer_name, learning_rate, weight_decay):
    if optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        return optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

In [17]:
def evaluate_c_index(model, X_eval, y_eval):
    model.eval()

    X_eval_tensor = torch.tensor(X_eval.values, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        preds = model(X_eval_tensor).cpu().numpy().reshape(-1)

    c_index = concordance_index(
        y_eval[DURATION_COL].values,
        -preds,
        y_eval[EVENT_COL].values
    )

    return float(c_index)

In [18]:
def train_one_fold(X_train_fold, y_train_fold, X_valid_fold, y_valid_fold, config):
    set_seed(RANDOM_STATE)

    X_train_tensor = torch.tensor(X_train_fold.values, dtype=torch.float32)
    y_train_duration = torch.tensor(y_train_fold[DURATION_COL].values, dtype=torch.float32)
    y_train_event = torch.tensor(y_train_fold[EVENT_COL].values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_duration, y_train_event)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    model = DeepSurv(
        input_dim=X_train_fold.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout_rate=config["dropout_rate"],
        activation_name=config["activation"]
    ).to(DEVICE)

    optimizer = build_optimizer(
        model=model,
        optimizer_name=config["optimizer"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    best_val_c_index = -np.inf
    best_epoch = 0
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(config["epochs"]):
        model.train()

        for batch_X, batch_duration, batch_event in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_duration = batch_duration.to(DEVICE)
            batch_event = batch_event.to(DEVICE)

            optimizer.zero_grad()
            preds = model(batch_X)
            loss = nll_loss(preds, batch_duration, batch_event)
            loss.backward()
            optimizer.step()

        val_c_index = evaluate_c_index(model, X_valid_fold, y_valid_fold)

        if val_c_index > best_val_c_index:
            best_val_c_index = val_c_index
            best_epoch = epoch + 1
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config["patience"]:
            break

    return best_val_c_index, best_epoch, best_state_dict

In [19]:
HIDDEN_LAYER_OPTIONS = [
    [64],
    [128],
    [128, 64],
    [256, 128]
]

DROPOUT_OPTIONS = [0.0, 0.2]
BATCH_SIZE_OPTIONS = [64, 128]
ACTIVATION_OPTIONS = ["ReLU", "LeakyReLU"]
OPTIMIZER_OPTIONS = ["Adam", "AdamW"]
WEIGHT_DECAY_OPTIONS = [0.0, 1e-4]
LEARNING_RATE_OPTIONS = [1e-4, 1e-3, 5e-3]
PATIENCE_OPTIONS = [10, 20]
EPOCH_OPTIONS = [75, 100]

In [20]:
all_configurations = list(itertools.product(
    HIDDEN_LAYER_OPTIONS,
    DROPOUT_OPTIONS,
    BATCH_SIZE_OPTIONS,
    ACTIVATION_OPTIONS,
    OPTIMIZER_OPTIONS,
    WEIGHT_DECAY_OPTIONS,
    LEARNING_RATE_OPTIONS,
    PATIENCE_OPTIONS,
    EPOCH_OPTIONS
))

print("Total grid combinations:", len(all_configurations))

Total grid combinations: 1536


In [21]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

trial_results = []
fold_details = []

In [22]:
for trial_counter, config_tuple in enumerate(all_configurations, start=1):
    (
        hidden_layers,
        dropout_rate,
        batch_size,
        activation,
        optimizer_name,
        weight_decay,
        learning_rate,
        patience,
        epochs
    ) = config_tuple

    config = {
        "hidden_layers": hidden_layers,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs
    }

    print(f"\nRunning trial {trial_counter}/{len(all_configurations)}")
    print(config)

    fold_scores = []
    fold_best_epochs = []

    for fold_idx, (train_index, valid_index) in enumerate(
        skf.split(X, y[EVENT_COL].astype(int).values), start=1
    ):
        X_train_fold = X.iloc[train_index].copy()
        X_valid_fold = X.iloc[valid_index].copy()

        y_train_fold = y.iloc[train_index].copy()
        y_valid_fold = y.iloc[valid_index].copy()

        try:
            best_val_c_index, best_epoch, _ = train_one_fold(
                X_train_fold,
                y_train_fold,
                X_valid_fold,
                y_valid_fold,
                config
            )

            fold_scores.append(float(best_val_c_index))
            fold_best_epochs.append(int(best_epoch))

            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "success",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": best_epoch,
                "c_index": float(best_val_c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "failed",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": np.nan,
                "c_index": np.nan,
                "error_message": str(e)
            })

    successful_folds = len(fold_scores)
    failed_folds = N_SPLITS - successful_folds

    trial_results.append({
        "trial": trial_counter,
        "hidden_layers": json.dumps(hidden_layers),
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs,
        "successful_folds": successful_folds,
        "failed_folds": failed_folds,
        "mean_best_epoch": np.mean(fold_best_epochs) if successful_folds > 0 else np.nan,
        "mean_c_index": np.mean(fold_scores) if successful_folds > 0 else np.nan,
        "std_c_index": np.std(fold_scores) if successful_folds > 0 else np.nan,
        "status": "ok" if successful_folds > 0 else "all_failed"
    })

    if successful_folds > 0:
        print("Fold C-indexes:", [round(score, 4) for score in fold_scores])
        print("Mean C-index:", round(np.mean(fold_scores), 4))
    else:
        print("All folds failed for this trial.")

Streaming output truncated to the last 5000 lines.

Running trial 537/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 75}
Fold C-indexes: [0.7171, 0.7256, 0.7431, 0.7129, 0.7221]
Mean C-index: 0.7242

Running trial 538/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 100}
Fold C-indexes: [0.7171, 0.7256, 0.7431, 0.7129, 0.7221]
Mean C-index: 0.7242

Running trial 539/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 20, 'epochs': 75}
Fold C-indexes: [0.7171, 0.7256, 0.7431, 0.7143, 0.7221]
Mean C-index: 0.7245

Running trial 540/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_s

In [23]:
results_df = pd.DataFrame(trial_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

fold_details_df = pd.DataFrame(fold_details)

results_df.head(10)

,trial,hidden_layers,dropout_rate,batch_size,activation,optimizer,weight_decay,learning_rate,patience,epochs,successful_folds,failed_folds,mean_best_epoch,mean_c_index,std_c_index,status
0,923,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0000,0.005,20,75,5,0,47.4,0.725772,0.007802,ok
1,924,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0000,0.005,20,100,5,0,47.4,0.725772,0.007802,ok
2,947,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0000,0.005,20,75,5,0,47.4,0.725772,0.007802,ok
3,948,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0000,0.005,20,100,5,0,47.4,0.725772,0.007802,ok
4,959,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0001,0.005,20,75,5,0,44.8,0.725759,0.007860,ok
5,960,"[128, 64]",0.0,128,LeakyReLU,AdamW,0.0001,0.005,20,100,5,0,44.8,0.725759,0.007860,ok
6,624,[128],0.2,64,ReLU,AdamW,0.0001,0.005,20,100,5,0,53.0,0.725649,0.004802,ok
7,1272,"[256, 128]",0.0,128,ReLU,Adam,0.0001,0.005,20,100,5,0,55.0,0.725628,0.008086,ok
8,936,"[128, 64]",0.0,128,LeakyReLU,Adam,0.0001,0.005,20,100,5,0,59.2,0.725626,0.007884,ok
9,1271,"[256, 128]",0.0,128,ReLU,Adam,0.0001,0.005,20,75,5,0,49.4,0.725543,0.008157,ok


In [24]:
results_df.to_csv(TUNING_RESULTS_PATH, index=False)
fold_details_df.to_csv(TUNING_FOLD_DETAILS_PATH, index=False)

TUNING_RESULTS_PATH, TUNING_FOLD_DETAILS_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_deepsurv_tuning_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_deepsurv_tuning_fold_details.csv')

In [25]:
valid_results = results_df.dropna(subset=["mean_c_index"]).copy()

if valid_results.empty:
    raise RuntimeError("All DeepSurv CV configurations failed. Check TUNING_FOLD_DETAILS_PATH for errors.")

best_result = valid_results.iloc[0].copy()
best_hidden_layers = json.loads(best_result["hidden_layers"])

BEST_HYPERPARAMETERS = {
    "hidden_layers": best_hidden_layers,
    "dropout_rate": float(best_result["dropout_rate"]),
    "batch_size": int(best_result["batch_size"]),
    "activation": str(best_result["activation"]),
    "optimizer": str(best_result["optimizer"]),
    "weight_decay": float(best_result["weight_decay"]),
    "learning_rate": float(best_result["learning_rate"]),
    "patience": int(best_result["patience"]),
    "epochs": int(best_result["epochs"]),
    "mean_best_epoch": float(best_result["mean_best_epoch"]),
    "mean_c_index": float(best_result["mean_c_index"])
}

print("Best hyperparameters:")
print(BEST_HYPERPARAMETERS)

Best hyperparameters:
{'hidden_layers': [128, 64], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 20, 'epochs': 75, 'mean_best_epoch': 47.4, 'mean_c_index': 0.725771675236463}


In [26]:
with open(BEST_PARAMS_JSON_PATH, "w") as file:
    json.dump(BEST_HYPERPARAMETERS, file, indent=2)

best_params_df = pd.DataFrame([{
    "hidden_layers": json.dumps(BEST_HYPERPARAMETERS["hidden_layers"]),
    "dropout_rate": BEST_HYPERPARAMETERS["dropout_rate"],
    "batch_size": BEST_HYPERPARAMETERS["batch_size"],
    "activation": BEST_HYPERPARAMETERS["activation"],
    "optimizer": BEST_HYPERPARAMETERS["optimizer"],
    "weight_decay": BEST_HYPERPARAMETERS["weight_decay"],
    "learning_rate": BEST_HYPERPARAMETERS["learning_rate"],
    "patience": BEST_HYPERPARAMETERS["patience"],
    "epochs": BEST_HYPERPARAMETERS["epochs"],
    "mean_best_epoch": BEST_HYPERPARAMETERS["mean_best_epoch"],
    "mean_c_index": BEST_HYPERPARAMETERS["mean_c_index"]
}])

best_params_df.to_csv(BEST_PARAMS_CSV_PATH, index=False)

BEST_PARAMS_JSON_PATH, BEST_PARAMS_CSV_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_deepsurv_best_hyperparameters.json',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/exp_1/hcc_exp1_deepsurv_best_hyperparameters.csv')

In [27]:
print("Best hidden layers:", BEST_HYPERPARAMETERS["hidden_layers"])
print("Best dropout rate:", BEST_HYPERPARAMETERS["dropout_rate"])
print("Best batch size:", BEST_HYPERPARAMETERS["batch_size"])
print("Best activation:", BEST_HYPERPARAMETERS["activation"])
print("Best optimizer:", BEST_HYPERPARAMETERS["optimizer"])
print("Best weight decay:", BEST_HYPERPARAMETERS["weight_decay"])
print("Best learning rate:", BEST_HYPERPARAMETERS["learning_rate"])
print("Best patience:", BEST_HYPERPARAMETERS["patience"])
print("Best epochs:", BEST_HYPERPARAMETERS["epochs"])
print("Best mean validation C-index:", round(BEST_HYPERPARAMETERS["mean_c_index"], 4))

Best hidden layers: [128, 64]
Best dropout rate: 0.0
Best batch size: 128
Best activation: LeakyReLU
Best optimizer: Adam
Best weight decay: 0.0
Best learning rate: 0.005
Best patience: 20
Best epochs: 75
Best mean validation C-index: 0.7258


The rerun of the `Exp_1` DeepSurv training notebook completed successfully and produced a slightly updated best configuration, while preserving the same overall pattern we had already been seeing. The notebook again carried out the full cross-validated hyperparameter search on the training set only, ranking candidate models by mean validation C-index across the 5 folds. This means the rerun remained leakage-safe and the test set was still untouched during selection.

The strongest configuration from the rerun was:

Best hidden layers: [128, 64]
Best dropout rate: 0.0
Best batch size: 128
Best activation: LeakyReLU
Best optimizer: Adam
Best weight decay: 0.0
Best learning rate: 0.005
Best patience: 20
Best epochs: 75
Best mean validation C-index: 0.7258

Its final cross-validated performance was:
- Mean validation C-index: `0.7258`
- Mean best epoch: `47.4`
- Successful folds: `5/5`
- Failed folds: `0`

A key observation is that the rerun slightly shifted the winning setup compared with the earlier run. Previously, `Adam` with no weight decay had emerged at the top; in this rerun, `AdamW` with mild regularization (`weight_decay = 0.0001`) and `100` epochs came out best. But the change is small and the top trials are extremely close to one another. Many of the leading configurations cluster tightly around `0.7255` to `0.7258`, which tells us the model is converging to a very stable high-performing region of the search space rather than depending on one fragile lucky run.

Another important observation is that the overall structural conclusion stayed the same:
- the best architecture is still the two-layer network `[128, 64]`
- `LeakyReLU` remains the strongest activation
- dropout still does not help at the optimum
- learning rate `0.005` remains the best region
- larger patience continues to be beneficial

That consistency matters more than the tiny shifts in optimizer or regularization, because it shows the DeepSurv solution for `Exp_1` is reproducible in its main design choices.

In practical terms, this rerun confirms that `Exp_1` DeepSurv has a robust best-performing configuration and that the model-selection process is stable. The final takeaway is that the rerun slightly improves the verified best validation result to `0.7258`, and the correct configuration to carry forward is now the `AdamW`-based `[128, 64]` model with mild weight decay and `75` epochs.